In [31]:
%pip install -q transformers accelerate pillow requests matplotlib opencv-python fastapi uvicorn pyngrok

In [1]:
import os
import io
import time
import threading

import torch
import numpy as np
import cv2

from PIL import Image

print("Python environment ready.")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python environment ready.
PyTorch: 2.7.1+cu126
CUDA: True
GPU: Tesla T4


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
from transformers import Sam3Processor, Sam3Model

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading SAM3...")

processor = Sam3Processor.from_pretrained(
    "facebook/sam3"
)

model = Sam3Model.from_pretrained(
    "facebook/sam3",
    device_map="auto"
)

print("✅ SAM3 loaded successfully.")
print("Device:", device)

Loading SAM3...


[transformers] `memory_attention_rope_theta` is deprecated and will be removed in v5.0. Use `rope_parameters['rope_theta']` instead.


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

✅ SAM3 loaded successfully.
Device: cuda


In [4]:
print("processor:", processor is not None)
print("model:", model is not None)
print("device:", device)

processor: True
model: True
device: cuda


In [5]:
FACIAL_ZONES = [
    "forehead",
    "left cheek",
    "right cheek",
    "nose",
    "chin"
]

print("Facial zones:")

for zone in FACIAL_ZONES:
    print(" -", zone)

Facial zones:
 - forehead
 - left cheek
 - right cheek
 - nose
 - chin


In [6]:
def segment_zone(image, zone):

    inputs = processor(
        images=image,
        text=zone,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=0.5,
        mask_threshold=0.5,
        target_sizes=inputs["original_sizes"].tolist()
    )[0]

    return results


print("✅ segment_zone ready.")

✅ segment_zone ready.


In [7]:
def process_frame_with_sam3(image):

    output = np.array(
        image.convert("RGB")
    ).copy()

    zone_results = {}

    for zone in FACIAL_ZONES:

        print(f"SAM 3 → {zone}")

        results = segment_zone(
            image,
            zone
        )

        zone_results[zone] = results

        if len(results["masks"]) == 0:

            print(f"   No mask found: {zone}")

            continue

        masks = results["masks"].cpu().numpy()

        combined_mask = np.any(
            masks,
            axis=0
        )

        color = np.array(
            [0, 255, 0],
            dtype=np.uint8
        )

        output[combined_mask] = cv2.addWeighted(
            output[combined_mask],
            0.5,
            np.full_like(
                output[combined_mask],
                color
            ),
            0.5,
            0
        )

    return Image.fromarray(output), zone_results


print("✅ process_frame_with_sam3 ready.")

✅ process_frame_with_sam3 ready.


In [8]:
latest_frame = None

# Clean/original image for color analysis
latest_analysis_image = None

# SAM3 visualization
latest_sam3_image = None

# Actual SAM3 masks/results
latest_zone_results = {}

sam3_lock = threading.Lock()

print("✅ Shared state initialized.")

✅ Shared state initialized.


In [9]:
def sam3_worker():

    global latest_frame
    global latest_analysis_image
    global latest_sam3_image
    global latest_zone_results

    while True:

        with sam3_lock:

            if latest_frame is None:

                frame_to_process = None

            else:

                frame_to_process = latest_frame.copy()

                latest_frame = None

        if frame_to_process is None:

            time.sleep(0.05)

            continue

        print("\n==============================")
        print("🧠 SAM3 WORKER STARTED")
        print("==============================")

        try:

            processed_image, zone_results = (
                process_frame_with_sam3(
                    frame_to_process
                )
            )

            with sam3_lock:

                # Clean image
                latest_analysis_image = (
                    frame_to_process.copy()
                )

                # Visualized image
                latest_sam3_image = (
                    processed_image.copy()
                )

                # Actual masks
                latest_zone_results = (
                    zone_results
                )

            print("✅ SAM3 WORKER FINISHED")

        except Exception as e:

            print(
                "❌ SAM3 ERROR:",
                repr(e)
            )

In [10]:
worker_thread = threading.Thread(
    target=sam3_worker,
    daemon=True
)

worker_thread.start()

print("🚀 SAM3 worker running.")
print("Worker alive:", worker_thread.is_alive())

🚀 SAM3 worker running.
Worker alive: True


In [11]:
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import Response

app = FastAPI()


@app.get("/")
def home():

    return {
        "status": "online",
        "server": "SkinGPT",
        "runtime": "Colab BOB",
        "sam3": "loaded"
    }


@app.get("/debug")
def debug():

    return {
        "sam3_loaded": model is not None,
        "inference_has_run": (
            latest_sam3_image is not None
        ),
        "zones": list(
            latest_zone_results.keys()
        )
    }

In [12]:
@app.post("/frame")
async def process_frame(
    file: UploadFile = File(...)
):

    global latest_frame
    global latest_sam3_image

    image_bytes = await file.read()

    image = Image.open(
        io.BytesIO(image_bytes)
    ).convert("RGB")

    with sam3_lock:

        latest_frame = image

        if latest_sam3_image is not None:

            output_image = (
                latest_sam3_image.copy()
            )

        else:

            output_image = image.copy()

    buffer = io.BytesIO()

    output_image.save(
        buffer,
        format="JPEG",
        quality=85
    )

    return Response(
        content=buffer.getvalue(),
        media_type="image/jpeg"
    )

In [13]:
print("Registered routes:")

for route in app.routes:

    print(
        getattr(route, "methods", None),
        route.path
    )

Registered routes:
{'GET', 'HEAD'} /openapi.json
{'GET', 'HEAD'} /docs
{'GET', 'HEAD'} /docs/oauth2-redirect
{'GET', 'HEAD'} /redoc
{'GET'} /
{'GET'} /debug
{'POST'} /frame


In [ ]:
import uvicorn


def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("🚀 FastAPI server starting...")

🚀 FastAPI server starting...


INFO:     Started server process [28988]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [16]:
import requests

r = requests.get(
    "http://127.0.0.1:8000",
    timeout=5
)

print("Status:", r.status_code)
print("Response:", r.json())

INFO:     127.0.0.1:42156 - "GET / HTTP/1.1" 200 OK
Status: 200
Response: {'status': 'online', 'server': 'SkinGPT', 'runtime': 'Colab BOB', 'sam3': 'loaded'}


In [17]:
from pyngrok import ngrok

ngrok.kill()

ngrok.set_auth_token(
    "3JPPB6VmhvCY6YgNCaVxTAwLuKo_7mTgXFNSzF9vGmgfbGPpA"
)

tunnel = ngrok.connect(8000)

public_url = tunnel.public_url

print("SkinGPT URL:")
print(public_url)

print("\nFrame endpoint:")
print(
    public_url.rstrip("/") + "/frame"
)

SkinGPT URL:
https://chowder-groove-irritable.ngrok-free.dev

Frame endpoint:
https://chowder-groove-irritable.ngrok-free.dev/frame


In [18]:
r = requests.get(
    public_url,
    timeout=10
)

print("Status:", r.status_code)
print("Response:", r.json())

INFO:     35.203.173.3:0 - "GET / HTTP/1.1" 200 OK
Status: 200
Response: {'status': 'online', 'server': 'SkinGPT', 'runtime': 'Colab BOB', 'sam3': 'loaded'}


In [19]:
print("Worker alive:", worker_thread.is_alive())

with sam3_lock:

    print(
        "Latest analysis image:",
        latest_analysis_image is not None
    )

    print(
        "Latest SAM3 image:",
        latest_sam3_image is not None
    )

    print(
        "Zones:",
        list(latest_zone_results.keys())
    )

Worker alive: True
Latest analysis image: False
Latest SAM3 image: False
Zones: []


In [20]:
with sam3_lock:

    results_copy = dict(
        latest_zone_results
    )

for zone, results in results_copy.items():

    print(
        zone,
        "→",
        len(results["masks"]),
        "masks"
    )

In [23]:
with sam3_lock:

    analysis_image = (
        latest_analysis_image.copy()
        if latest_analysis_image is not None
        else None
    )

if analysis_image is None:

    print("❌ No analysis image yet.")

else:

    print(
        "✅ Analysis image:",
        analysis_image.size,
        analysis_image.mode
    )

✅ Analysis image: (640, 480) RGB


In [21]:
import os

os.makedirs("/content/analysis", exist_ok=True)

with open("/content/analysis/__init__.py", "w") as f:
    f.write("")

print("✅ Analysis package created")

✅ Analysis package created


INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: forehead
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: nose
SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POS

In [ ]:
%%writefile /content/analysis/skin_features.py

import numpy as np
import cv2

def extract_region_pixels(image, mask):
    mask = mask.astype(bool)
    pixels = image[mask]
    return pixels

def calculate_brightness(image, mask):
    pixels = extract_region_pixels(image, mask)

    if len(pixels) == 0:
        return None

    gray = cv2.cvtColor(
        pixels.reshape(-1, 1, 3).astype(np.uint8),
        cv2.COLOR_RGB2GRAY
    )

    return float(np.mean(gray))

def calculate_redness(image, mask):
    pixels = extract_region_pixels(image, mask)

    if len(pixels) == 0:
        return None

    pixels = pixels.astype(np.float32)

    r = pixels[:, 0]
    g = pixels[:, 1]
    b = pixels[:, 2]

    denominator = r + g + b + 1e-6

    redness = r / denominator

    return float(np.mean(redness))

def calculate_color_statistics(image, mask):
    pixels = extract_region_pixels(image, mask)

    if len(pixels) == 0:
        return None

    pixels = pixels.astype(np.float32)

    return {
        "mean_r": float(np.mean(pixels[:, 0])),
        "mean_g": float(np.mean(pixels[:, 1])),
        "mean_b": float(np.mean(pixels[:, 2])),
        "std_r": float(np.std(pixels[:, 0])),
        "std_g": float(np.std(pixels[:, 1])),
        "std_b": float(np.std(pixels[:, 2]))
    }

def calculate_hsv_statistics(image, mask):
    pixels = extract_region_pixels(image, mask)

    if len(pixels) == 0:
        return None

    pixels = pixels.astype(np.uint8)

    hsv = cv2.cvtColor(
        pixels.reshape(-1, 1, 3),
        cv2.COLOR_RGB2HSV
    ).reshape(-1, 3)

    return {
        "mean_hue": float(np.mean(hsv[:, 0])),
        "mean_saturation": float(np.mean(hsv[:, 1])),
        "mean_value": float(np.mean(hsv[:, 2])),
        "std_hue": float(np.std(hsv[:, 0])),
        "std_saturation": float(np.std(hsv[:, 1])),
        "std_value": float(np.std(hsv[:, 2]))
    }

def analyze_region(image, mask):
    return {
        "brightness": calculate_brightness(image, mask),
        "redness": calculate_redness(image, mask),
        "color_statistics": calculate_color_statistics(image, mask),
        "hsv_statistics": calculate_hsv_statistics(image, mask)
    }

Overwriting /content/analysis/skin_features.py


   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK


In [23]:
%%writefile /content/analysis/skin_profile.py

def build_skin_profile(
    zone_results,
    image
):
    from .skin_features import analyze_region

    profile = {}

    for zone, results in zone_results.items():

        if len(results["masks"]) == 0:
            profile[zone] = {
                "status": "no_mask"
            }
            continue

        masks = results["masks"].cpu().numpy()

        combined_mask = masks.any(axis=0)

        features = analyze_region(
            image,
            combined_mask
        )

        profile[zone] = {
            "status": "analyzed",
            "features": features
        }

    return profile

Overwriting /content/analysis/skin_profile.py


INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK


In [ ]:
%%writefile /content/analysis/mvp_pipeline.py

from .skin_profile import build_skin_profile

def run_color_analysis(image, zone_results):
    """
    Run the Skin-GPT MVP color-analysis pipeline.
    """

    profile = build_skin_profile(
        zone_results=zone_results,
        image=image
    )

    return {
        "status": "success",
        "analysis_type": "color",
        "zones": profile
    }

Overwriting /content/analysis/mvp_pipeline.py


INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK


In [25]:
import os

print("Analysis files:")
for file in os.listdir("/content/analysis"):
    print(" -", file)

Analysis files:
 - mvp_pipeline.py
 - skin_profile.py
 - skin_features.py
 - __pycache__
 - __init__.py


   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin


In [ ]:
from analysis.skin_features import analyze_region
from analysis.skin_profile import build_skin_profile
from analysis.mvp_pipeline import run_color_analysis

print("✅ Color analysis modules loaded")

✅ Color analysis modules loaded


INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK


In [ ]:
import numpy as np

with sam3_lock:
    image_for_analysis = latest_analysis_image.copy()
    zone_results_for_analysis = latest_zone_results.copy()

# Convert PIL Image → NumPy RGB array
image_for_analysis = np.array(
    image_for_analysis.convert("RGB")
)

print("Image shape:", image_for_analysis.shape)
print("Image dtype:", image_for_analysis.dtype)
print("Zones:", list(zone_results_for_analysis.keys()))

Image shape: (480, 640, 3)
Image dtype: uint8
Zones: ['forehead', 'left cheek', 'right cheek', 'nose', 'chin']


INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek


In [ ]:
result = run_color_analysis(
    image=image_for_analysis,
    zone_results=zone_results_for_analysis
)

print("✅ Color analysis complete")

INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
✅ Color analysis complete


INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
 

In [ ]:
with sam3_lock:
    zone_results_check = latest_zone_results.copy()

print("SAM3 zone results:")

for zone, results in zone_results_check.items():

    num_masks = len(results["masks"])

    print(f"{zone}: {num_masks} mask(s)")

SAM3 zone results:
forehead: 1 mask(s)
left cheek: 0 mask(s)
right cheek: 0 mask(s)
nose: 1 mask(s)
chin: 1 mask(s)


✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
✅

In [31]:
import numpy as np

with sam3_lock:
    image_for_analysis = latest_analysis_image.copy()
    zone_results_for_analysis = latest_zone_results.copy()

image_for_analysis = np.array(
    image_for_analysis.convert("RGB")
)

# Keep only zones where SAM3 actually found a mask
valid_zone_results = {
    zone: results
    for zone, results in zone_results_for_analysis.items()
    if len(results["masks"]) > 0
}

print("Available zones:")
for zone, results in valid_zone_results.items():
    print(f"  ✅ {zone}: {len(results['masks'])} mask(s)")

print("\nSkipped zones:")
for zone, results in zone_results_for_analysis.items():
    if len(results["masks"]) == 0:
        print(f"  ⏭️ {zone}")

Available zones:
  ✅ forehead: 1 mask(s)
  ✅ nose: 1 mask(s)
  ✅ chin: 1 mask(s)

Skipped zones:
  ⏭️ left cheek
  ⏭️ right cheek


SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: chin
✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin
INFO:     31.209.42.201

In [ ]:
result = run_color_analysis(
    image=image_for_analysis,
    zone_results=valid_zone_results
)

print("✅ Color analysis complete")

INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
✅ Color analysis complete


SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin
INFO:     31.209.42.201:0 - "POST /frame HTTP/

In [ ]:
print(result)

{'status': 'success', 'analysis_type': 'color', 'zones': {'forehead': {'status': 'analyzed', 'features': {'brightness': 153.0171452179302, 'redness': 0.385191410779953, 'color_statistics': {'mean_r': 177.65048217773438, 'mean_g': 142.34869384765625, 'mean_b': 143.19004821777344, 'std_r': 45.126766204833984, 'std_g': 42.44991683959961, 'std_b': 39.610347747802734}, 'hsv_statistics': {'mean_hue': 97.07973559181987, 'mean_saturation': 56.108861805412104, 'mean_value': 177.6504854368932, 'std_hue': 84.60976587438493, 'std_saturation': 12.938013675119555, 'std_value': 45.12676863160709}}}, 'nose': {'status': 'analyzed', 'features': {'brightness': 214.52190476190475, 'redness': 0.36269569396972656, 'color_statistics': {'mean_r': 231.9117431640625, 'mean_g': 206.95111083984375, 'mean_b': 207.77841186523438, 'std_r': 31.917699813842773, 'std_g': 45.5606689453125, 'std_b': 44.01168441772461}, 'hsv_statistics': {'mean_hue': 92.20380952380953, 'mean_saturation': 32.42349206349206, 'mean_value': 2

   No mask found: right cheek
SAM 3 → nose
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.42.201:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → chin
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
✅ SAM3 WORKER FINISHED

🧠 SAM3 WORKER STARTED
SAM 3 → forehead
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
SAM 3 → left cheek
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: left cheek
SAM 3 → right cheek
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
INFO:     31.209.41.226:0 - "POST /frame HTTP/1.1" 200 OK
   No mask found: right cheek
SAM 3 → nose
INFO:     31.209

In [6]:
import os

print(os.getcwd())
print(os.listdir("/content"))

/content
['.config', 'sample_data']
